# Part II · Lesson 2 — Four ways to connect a graph

## Shared imports

### Import the graph helpers

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

## 1. Sequential — pizza order pipeline

### Describe the pizza state

In [ ]:
class PizzaState(TypedDict):
    items: list[str]
    total: int
    status: str

### Prepare a menu and an order

In [ ]:
menu = {"Margherita": 250, "Farmhouse": 350}
pizza_input = {
    "items": ["Margherita", "Farmhouse"],
    "total": 0,
    "status": "new",
}

### Validate the order

In [ ]:
def validate_order(state: PizzaState) -> dict:
    if not state["items"] or any(item not in menu for item in state["items"]):
        raise ValueError("Choose at least one pizza from the menu.")
    return {"status": "validated"}

### Calculate the total

In [ ]:
def calculate_total(state: PizzaState) -> dict:
    total = sum(menu[item] for item in state["items"])
    return {"total": total}

### Apply the coupon

In [ ]:
def apply_coupon(state: PizzaState) -> dict:
    return {"total": state["total"] * 90 // 100}

### Confirm the discounted order

In [ ]:
def confirm(state: PizzaState) -> dict:
    return {"status": f"Confirmed at INR {state['total']}"}

### Register the pizza nodes

In [ ]:
sequential_builder = StateGraph(PizzaState)
sequential_builder.add_node("validate_order", validate_order)
sequential_builder.add_node("calculate_total", calculate_total)
sequential_builder.add_node("apply_coupon", apply_coupon)
sequential_builder.add_node("confirm", confirm)

### Connect the pipeline

In [ ]:
sequential_builder.add_edge(START, "validate_order")
sequential_builder.add_edge("validate_order", "calculate_total")
sequential_builder.add_edge("calculate_total", "apply_coupon")
sequential_builder.add_edge("apply_coupon", "confirm")
sequential_builder.add_edge("confirm", END)
pizza_graph = sequential_builder.compile()

### Run the pizza order

In [ ]:
pizza_result = pizza_graph.invoke(pizza_input)
print(pizza_result)

## 2. Parallel — trip dashboard

### Describe the trip state

In [ ]:
import time

class TripState(TypedDict):
    city: str
    messages: Annotated[list, add_messages]
    report: str

### Plan the trip

In [ ]:
def plan(state: TripState) -> dict:
    return {"messages": [f"Planning a trip to {state['city']}."]}

### Look up the weather

In [ ]:
def get_weather(state: TripState) -> dict:
    time.sleep(1)
    return {"messages": [f"Weather in {state['city']}: sunny, 28 C."]}

### Look up flights

In [ ]:
def get_flights(state: TripState) -> dict:
    time.sleep(1)
    return {"messages": [f"Flights to {state['city']}: 3 sample options."]}

### Look up hotels

In [ ]:
def get_hotels(state: TripState) -> dict:
    time.sleep(1)
    return {"messages": [f"Hotels in {state['city']}: 5 sample options."]}

### Combine the lookup results

In [ ]:
def combine(state: TripState) -> dict:
    report = "\n".join(message.content for message in state["messages"])
    return {"report": report}

### Register the trip nodes

In [ ]:
parallel_builder = StateGraph(TripState)
parallel_builder.add_node("plan", plan)
parallel_builder.add_node("get_weather", get_weather)
parallel_builder.add_node("get_flights", get_flights)
parallel_builder.add_node("get_hotels", get_hotels)
parallel_builder.add_node("combine", combine)

### Fan out and wait for all three lookups

In [ ]:
parallel_builder.add_edge(START, "plan")
parallel_builder.add_edge("plan", "get_weather")
parallel_builder.add_edge("plan", "get_flights")
parallel_builder.add_edge("plan", "get_hotels")
# Wait for every lookup before combining the results.
parallel_builder.add_edge(["get_weather", "get_flights", "get_hotels"], "combine")
parallel_builder.add_edge("combine", END)
trip_graph = parallel_builder.compile()

### Run and time the dashboard

In [ ]:
trip_input = {"city": "Goa", "messages": [], "report": ""}
started = time.perf_counter()
trip_result = trip_graph.invoke(trip_input)
elapsed = time.perf_counter() - started
print(trip_result["report"])
print(f"Elapsed: {elapsed:.2f} seconds")

## 3. Conditional — support ticket router

### Describe the ticket state

In [ ]:
class TicketState(TypedDict):
    text: str
    type: str
    response: str

### Classify the ticket

In [ ]:
def classify(state: TicketState) -> dict:
    text = state["text"].lower()
    if "urgent" in text:
        ticket_type = "urgent"
    elif "bill" in text:
        ticket_type = "billing"
    else:
        ticket_type = "general"
    print(f"classify -> {ticket_type}")
    return {"type": ticket_type}

### Define the three possible handlers

In [ ]:
def urgent_handler(state: TicketState) -> dict:
    print("urgent_handler")
    return {"response": "Queued for human review."}

def billing_handler(state: TicketState) -> dict:
    print("billing_handler")
    return {"response": "Sent to the billing queue."}

def general_handler(state: TicketState) -> dict:
    print("general_handler")
    return {"response": "Sent to the general support queue."}

### Read the category to choose a path

In [ ]:
def route(state: TicketState) -> str:
    return state["type"]  # "urgent", "billing", or "general"

### Register classification and the handlers

In [ ]:
conditional_builder = StateGraph(TicketState)
conditional_builder.add_node("classify", classify)
conditional_builder.add_node("urgent_handler", urgent_handler)
conditional_builder.add_node("billing_handler", billing_handler)
conditional_builder.add_node("general_handler", general_handler)

### Connect the three conditional routes

In [ ]:
conditional_builder.add_edge(START, "classify")
conditional_builder.add_conditional_edges(
    "classify", route,
    {"urgent": "urgent_handler", "billing": "billing_handler", "general": "general_handler"},
)
conditional_builder.add_edge("urgent_handler", END)
conditional_builder.add_edge("billing_handler", END)
conditional_builder.add_edge("general_handler", END)
ticket_graph = conditional_builder.compile()

### Run one ticket for each category

In [ ]:
ticket_inputs = ["Urgent: the site is down", "My bill is incorrect", "How do I change my name?"]
for text in ticket_inputs:
    print(f"\nTicket: {text}")
    result = ticket_graph.invoke({"text": text, "type": "", "response": ""})
    print(result["response"])

## 4. Loop — OTP retry with a limit

### Describe the OTP state

In [ ]:
class OtpState(TypedDict):
    entries: list[str]
    otp: str
    tries: int
    verified: bool
    status: str

### Take the next sample entry

In [ ]:
def ask_otp(state: OtpState) -> dict:
    entries = state["entries"]
    # Repeat the last sample entry if the list runs out.
    otp = entries[state["tries"]] if state["tries"] < len(entries) else entries[-1]
    tries = state["tries"] + 1
    print(f"ask_otp: attempt {tries}")
    return {"otp": otp, "tries": tries}

### Verify the submitted code

In [ ]:
def verify(state: OtpState) -> dict:
    verified = state["otp"] == "123456"
    print(f"verify: {verified}")
    return {"verified": verified}

### Choose success, retry, or lockout

In [ ]:
def otp_route(state: OtpState) -> str:
    if state["verified"]:
        return "correct"
    if state["tries"] >= 3:
        return "locked"
    return "retry"

### Record the two final outcomes

In [ ]:
def success(state: OtpState) -> dict:
    return {"status": "verified"}

def locked(state: OtpState) -> dict:
    return {"status": "locked"}

### Register the OTP nodes

In [ ]:
loop_builder = StateGraph(OtpState)
loop_builder.add_node("ask_otp", ask_otp)
loop_builder.add_node("verify", verify)
loop_builder.add_node("success", success)
loop_builder.add_node("locked", locked)

### Connect the retry edge and both exits

In [ ]:
loop_builder.add_edge(START, "ask_otp")
loop_builder.add_edge("ask_otp", "verify")
loop_builder.add_conditional_edges(
    "verify", otp_route,
    {"correct": "success", "retry": "ask_otp", "locked": "locked"},
)
loop_builder.add_edge("success", END)
loop_builder.add_edge("locked", END)
otp_graph = loop_builder.compile()

### Try a wrong entry followed by a correct one

In [ ]:
otp_input = {
    "entries": ["000000", "123456"], "otp": "",
    "tries": 0, "verified": False, "status": "new",
}
otp_result = otp_graph.invoke(otp_input, {"recursion_limit": 12})
print(otp_result["tries"], otp_result["status"])

### Try repeated wrong entries

In [ ]:
locked_input = {**otp_input, "entries": ["000000"]}
locked_result = otp_graph.invoke(locked_input, {"recursion_limit": 12})
print(locked_result["tries"], locked_result["status"])

### Optional experiment — remove the normal retry limit

### Observe the graph-step limit

In [ ]:
# Optional: run only after removing the attempt limit and rebuilding the graph.
# otp_graph.invoke(locked_input, {"recursion_limit": 6})